# Notebook 00 -- End-to-End Smoke Test

**Research:** Hybrid AO-PSO Optimised Temporal Fusion Transformer for Smart Grid Multi-Horizon Energy Forecasting

---

## Purpose

This notebook validates that the entire `src/` pipeline executes end-to-end **before** committing to expensive GPU runs on Vast.ai. It is designed to complete in under 60 seconds on GPU (or ~2-3 minutes on CPU).

**What is being verified:**

| Step | Module | Validation |
|------|--------|------------|
| 1 | `dataset_utils.py` | CSV -> preprocess -> split -> scale -> DataLoaders (tensor shapes) |
| 2 | `optimizers.py` | `ObjectiveFunction` instantiation + `Hybrid_AO_PSO` hand-off mechanism |
| 3 | `models.py` | `BaseTFT` forward pass with optimizer-discovered hyperparameters |
| 4 | `metrics.py` | Inverse scaling + RMSE / MAE / MAPE calculation on real-world units |

**Smoke test settings** (intentionally minimal):
- `pop_size = 4`, `total_iter = 4` (2 AO + 2 PSO)
- `proxy_epochs = 1`, `subset_fraction = 0.05`, `val_subset_fraction = 0.1`
- Dataset: Micro-Grid (UCI Household)

In [ ]:
%pip install numpy pandas torch matplotlib scikit-learn scipy xgboost

In [ ]:
# ================================================================
# Cell 1: Imports
# ================================================================
import sys
import os
import logging

import numpy as np
import torch
import matplotlib.pyplot as plt

# ---- Set project root ----
PROJECT_ROOT = "/workspace/Hybrid-Optimizer-AO-PSO" 

assert os.path.isdir(os.path.join(PROJECT_ROOT, "src")), (
    f"src/ not found in {PROJECT_ROOT}. "
    f"Update PROJECT_ROOT to your actual project path on this machine."
)

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

from src.dataset_utils import prepare_pipeline
from src.models import BaseTFT
from src.optimizers import Hybrid_AO_PSO, ObjectiveFunction, decode_position
from src.metrics import (
    inverse_transform_predictions,
    calculate_forecasting_metrics,
)

# Enable optimizer logging so we can see the hand-off in real time
logging.basicConfig(level=logging.INFO, format="%(message)s")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")

: 

In [ ]:
# ================================================================
# Cell 2: Data Pipeline
# ================================================================
# Load the Micro-Grid (UCI Household) dataset through the full
# pipeline: CSV -> preprocess -> chronological split -> scale -> DataLoaders

pipeline = prepare_pipeline(
    filepath="data/micro_household_dataset.csv",
    dataset_type="micro",
    window_size=168,
    horizon=24,
    batch_size=64,
    num_workers=4,  # Low worker count to minimise process-spawn overhead
)

# Unpack the artifacts we need
train_loader = pipeline["train_loader"]
val_loader   = pipeline["val_loader"]
test_loader  = pipeline["test_loader"]
scaler       = pipeline["scaler"]
target_idx   = pipeline["target_idx"]
n_features   = pipeline["n_continuous"]
window_size  = pipeline["window_size"]
horizon      = pipeline["horizon"]

# Quick shape sanity check on one batch
x_cont, x_cat, y = next(iter(train_loader))
print(f"\nShape check:")
print(f"  x_cont: {x_cont.shape}  (B, T, F)")
print(f"  x_cat:  {x_cat.shape}   (B, T, C)")
print(f"  y:      {y.shape}       (B, H)")
print(f"  n_features (continuous): {n_features}")
print(f"  target_idx: {target_idx} -> '{pipeline['target_col']}'")

In [ ]:
# ================================================================
# Cell 3: Objective Function
# ================================================================
# Instantiate the fitness evaluator with minimal settings so each
# candidate evaluation takes ~1-2 seconds instead of minutes.
#   proxy_epochs=1          -> train each candidate for just 1 epoch
#   subset_fraction=0.05    -> use only 5% of training batches
#   val_subset_fraction=0.1 -> use only 10% of validation batches

objective_fn = ObjectiveFunction(
    train_loader=train_loader,
    val_loader=val_loader,
    n_encoder_features=n_features,
    window_size=window_size,
    horizon=horizon,
    proxy_epochs=1,
    subset_fraction=0.05,
    val_subset_fraction=0.1,
    device=DEVICE,
)

print("ObjectiveFunction ready.")

In [ ]:
# ================================================================
# Cell 4: Hybrid AO-PSO Smoke Run
# ================================================================
# Minimal configuration to prove the hand-off mechanism works:
#   pop_size=4    -> tiny swarm (4 agents)
#   total_iter=4  -> 2 AO iterations + 2 PSO iterations
#   ao_fraction=0.5 -> 50/50 split between phases

hybrid = Hybrid_AO_PSO(
    objective_fn=objective_fn,
    pop_size=4,
    total_iter=4,
    ao_fraction=0.5,
    seed=42,
)

best_params, best_fitness, convergence = hybrid.optimize()

print("\n" + "=" * 60)
print("  SMOKE TEST: Hybrid AO-PSO Complete")
print("=" * 60)
print(f"  Best Validation RMSE: {best_fitness:.6f}")
for k, v in best_params.items():
    print(f"    {k:22s}: {v}")
print("=" * 60)

In [ ]:
# ================================================================
# Cell 5: Convergence Curve (Publication Quality)
# ================================================================
# Extract convergence history from the unified ConvergenceRecord.
# Each record has 'iteration', 'best_fitness', 'phase' keys.

history = convergence.history
iterations   = [r["iteration"] for r in history]
best_fitness_curve = [r["best_fitness"] for r in history]
phases       = [r["phase"] for r in history]

# Identify the hand-off point (last AO iteration)
ao_iters = [r["iteration"] for r in history if r["phase"] == "AO"]
handoff_iter = max(ao_iters) if ao_iters else 0

# -- Plot --
fig, ax = plt.subplots(figsize=(10, 5), dpi=150)

# Separate AO and PSO points for distinct markers
ao_x = [r["iteration"] for r in history if r["phase"] == "AO"]
ao_y = [r["best_fitness"] for r in history if r["phase"] == "AO"]
pso_x = [r["iteration"] for r in history if r["phase"] == "PSO"]
pso_y = [r["best_fitness"] for r in history if r["phase"] == "PSO"]

ax.plot(ao_x, ao_y, "o-", color="#2196F3", markersize=7,
        linewidth=2, label="Phase 1: AO (Global Exploration)", zorder=3)
ax.plot(pso_x, pso_y, "s-", color="#E91E63", markersize=7,
        linewidth=2, label="Phase 2: PSO (Local Exploitation)", zorder=3)

# Hand-off line
ax.axvline(x=handoff_iter + 0.5, color="#333333", linestyle="--",
           linewidth=1.5, alpha=0.7, label="Hand-Off Point")
ax.annotate("HAND-OFF", xy=(handoff_iter + 0.5, ax.get_ylim()[1]),
            xytext=(handoff_iter + 0.6, max(best_fitness_curve) * 0.98),
            fontsize=9, fontweight="bold", color="#333333",
            ha="left", va="top")

ax.set_xlabel("Iteration", fontsize=12, fontweight="bold")
ax.set_ylabel("Best Validation RMSE (scaled)", fontsize=12, fontweight="bold")
ax.set_title("Hybrid AO-PSO Convergence (Smoke Test)",
             fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=10, framealpha=0.9)
ax.grid(True, alpha=0.3, linestyle="-")
ax.set_xticks(iterations)

fig.tight_layout()
plt.show()

In [ ]:
# ================================================================
# Cell 6: Inference & Metrics Validation
# ================================================================
# Instantiate a fresh BaseTFT with the winning hyperparameters,
# run ONE batch through it, and validate the metrics pipeline.

model = BaseTFT(
    n_encoder_features=n_features,
    n_decoder_features=0,
    n_static_categoricals=0,
    d_model=best_params["d_model"],
    n_heads=best_params["n_heads"],
    num_encoder_layers=best_params["num_encoder_layers"],
    dropout=best_params["dropout"],
    horizon=horizon,
    window_size=window_size,
).to(DEVICE)

model.eval()

# Grab one batch from the validation loader
x_cont, x_cat, y_true_scaled = next(iter(val_loader))
x_cont = x_cont.to(DEVICE)

with torch.no_grad():
    y_pred_scaled = model(x_cont).cpu().numpy()

y_true_scaled = y_true_scaled.numpy()

print(f"Prediction shape: {y_pred_scaled.shape}  (B, H)")
print(f"Target shape:     {y_true_scaled.shape}  (B, H)")

# Inverse transform to original units
y_pred_real, y_true_real = inverse_transform_predictions(
    preds=y_pred_scaled,
    targets=y_true_scaled,
    scaler=scaler,
    target_idx=target_idx,
)

# Calculate metrics
metrics = calculate_forecasting_metrics(y_true_real, y_pred_real)

print("\n" + "=" * 60)
print("  METRICS VALIDATION (single batch, untrained model)")
print("=" * 60)
print(f"  RMSE : {metrics['RMSE']:.4f} {pipeline['target_col']}")
print(f"  MAE  : {metrics['MAE']:.4f} {pipeline['target_col']}")
print(f"  MAPE : {metrics['MAPE']:.2f}%")
print("=" * 60)
print("\nNote: High errors are expected -- this model is untrained.")
print("The purpose is to verify that all shapes, scaling, and metric")
print("calculations execute without errors.")
print("\n>>> SMOKE TEST PASSED <<<")